In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementation for circuit analysis in the repository:
`/net/scratch2/smallyan/function_vectors_eval`

## Setup and Overview

## Code Evaluation Process

Based on the Plan and CodeWalkthrough files, the main analysis code is contained in:
- `notebooks/fv_demo.ipynb` - Main demo notebook demonstrating function vector extraction and intervention
- `src/utils/*.py` - Utility modules supporting the analysis

The code implements the Function Vectors methodology from the ICLR 2024 paper, which:
1. Applies causal mediation analysis to identify attention heads with highest average indirect effect
2. Extracts function vectors by summing task-conditioned mean outputs of top causal attention heads
3. Tests function vectors across various contexts (ICL, shuffled-label, zero-shot, natural text)

In [2]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA H200 NVL
GPU Memory: 150.1 GB


In [3]:
# Set up paths
import os
import sys

REPO_PATH = '/net/scratch2/smallyan/function_vectors_eval'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

print(f"Working directory: {os.getcwd()}")

Working directory: /net/scratch2/smallyan/function_vectors_eval


## Code Block Evaluation

We will now run each code block from fv_demo.ipynb and evaluate:
1. **Runnable (Y/N)** - Does it execute without error?
2. **Correct-Implementation (Y/N)** - Is the logic correct per the stated purpose?
3. **Redundant (Y/N)** - Does it duplicate computation without new information?
4. **Irrelevant (Y/N)** - Does it contribute to the project goal?

### Block 1: Import and Setup

In [4]:
# Block 1: Autoreload setup (from fv_demo.ipynb cell 0)
block_1_status = {}
try:
    %load_ext autoreload
    %autoreload 2
    block_1_status['runnable'] = 'Y'
    block_1_status['error'] = None
except Exception as e:
    block_1_status['runnable'] = 'N'
    block_1_status['error'] = str(e)

print(f"Block 1 (Autoreload Setup): Runnable = {block_1_status['runnable']}")
if block_1_status['error']:
    print(f"Error: {block_1_status['error']}")

Block 1 (Autoreload Setup): Runnable = Y


In [5]:
# Block 2: Main imports (from fv_demo.ipynb cell 1)
block_2_status = {}
try:
    import os, re, json
    import torch, numpy as np

    import sys
    sys.path.append('..')
    torch.set_grad_enabled(False)

    from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
    from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
    from src.utils.model_utils import load_gpt_model_and_tokenizer
    from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
    from src.utils.eval_utils import decode_to_vocab, sentence_eval
    
    block_2_status['runnable'] = 'Y'
    block_2_status['error'] = None
except Exception as e:
    block_2_status['runnable'] = 'N'
    block_2_status['error'] = str(e)

print(f"Block 2 (Main Imports): Runnable = {block_2_status['runnable']}")
if block_2_status['error']:
    print(f"Error: {block_2_status['error']}")

Block 2 (Main Imports): Runnable = Y


In [6]:
# Block 3: Load model & tokenizer (from fv_demo.ipynb cell 3)
block_3_status = {}
try:
    model_name = 'EleutherAI/gpt-j-6b'
    model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
    EDIT_LAYER = 9
    
    block_3_status['runnable'] = 'Y'
    block_3_status['error'] = None
    print(f"Model loaded successfully: {model_name}")
    print(f"Model config: n_layers={model_config['n_layers']}, n_heads={model_config['n_heads']}, resid_dim={model_config['resid_dim']}")
except Exception as e:
    block_3_status['runnable'] = 'N'
    block_3_status['error'] = str(e)

print(f"\nBlock 3 (Load Model): Runnable = {block_3_status['runnable']}")
if block_3_status['error']:
    print(f"Error: {block_3_status['error']}")

Loading:  EleutherAI/gpt-j-6b


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

Model loaded successfully: EleutherAI/gpt-j-6b
Model config: n_layers=28, n_heads=16, resid_dim=4096

Block 3 (Load Model): Runnable = Y
